# 04 — NFL Outcome Inventory + Hypothesis Discovery

**Purpose**:
1. Profile available NFL outcome metrics from player_play.
2. Link Combine movement features to NFL outcomes.
3. Rank hypothesis candidates.
4. Generate `outputs/reports/hypothesis_candidates.md`.

**Temporal boundary**: Combine features are prospective (pre-draft).
NFL outcomes are from the player's FIRST NFL season only.
This is the correct scouting interpretation.

> **Critical**: Do not claim causation. Report associations with appropriate caveats.

In [ ]:
import sys
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ensure_dirs, FIGURES_DIR, TABLES_DIR, REPORTS_DIR
from src.config import COMBINE_FEATURES_PARQUET, PLAYER_MASTER_PARQUET, DATA_PROCESSED, MIN_SAMPLE_SIZE
from src.data_loading import load_players, load_combine_results, load_player_play
from src.player_mapping import (
    assign_position_groups, build_master_player_table,
    filter_to_first_season, compute_snap_counts,
)
from src.nfl_features import (
    aggregate_nfl_metrics, wr_te_metrics, dl_edge_metrics, ol_metrics, db_metrics,
)
from src.combine_features import aggregate_player_features
from src.statistics import (
    spearman_with_ci, build_correlation_summary, compare_groups,
    cohens_d, rank_biserial_r, incremental_r_squared,
)
from src.visualization import (
    plot_combine_vs_nfl, plot_correlation_heatmap,
    plot_model_comparison, plot_missing_data_heatmap,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()
print('Setup complete.')

## 1. Load All Processed Data

In [ ]:
# Load players
players = load_players()
players = assign_position_groups(players)
pos_map = players[['nfl_id', 'position', 'position_group']].drop_duplicates('nfl_id')

# Load combine features
if COMBINE_FEATURES_PARQUET.exists():
    features_df = pd.read_parquet(COMBINE_FEATURES_PARQUET)
    player_features = aggregate_player_features(features_df, agg_strategy='best_attempt')
    player_features = player_features.merge(pos_map, on='nfl_id', how='left')
    print(f'Combine features: {player_features.shape}')
else:
    print('WARNING: Run notebook 02 first to generate combine_features.parquet')
    player_features = pd.DataFrame()

# Load player-play data (NFL outcomes)
player_play = load_player_play()
print(f'player_play: {len(player_play):,} rows × {player_play.shape[1]} cols')
print(f'Unique players in player_play: {player_play["nfl_id"].nunique()}')

## 2. NFL Outcome Profiling

Understand what metrics are available, how sparse they are, and what values are typical.

In [ ]:
print('=== PLAYER PLAY — OUTCOME COLUMNS ===')
print(f'Available columns: {player_play.columns.tolist()}')
print()

# Missing rates for all potential outcome columns
outcome_candidates = [
    'target', 'rec_yards', 'yards_after_catch', 'separation_at_pass_forward',
    'cushion', 'expected_yards_after_catch', 'route_ran',
    'player_get_off', 'sack', 'tackle', 'tackle_for_loss',
    'time_to_pressure', 'quick_pressure',
    'pressure_allowed', 'sack_allowed', 'time_to_pressure_allowed',
]
available_outcomes = [c for c in outcome_candidates if c in player_play.columns]
print(f'Outcome columns found in player_play: {available_outcomes}')
print()

miss_pct = (player_play[available_outcomes].isnull().sum() / len(player_play) * 100).sort_values(ascending=False)
print('Missing rates (%) for outcome columns:')
display(miss_pct.to_frame('missing_pct'))

plot_missing_data_heatmap(
    player_play[available_outcomes],
    title='Missing Data — NFL Outcome Columns',
    filename='missing_nfl_outcomes.png',
)
plt.show()

In [ ]:
# Numeric summary of outcome columns
print('Numeric summary of NFL outcome columns:')
display(player_play[available_outcomes].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

## 3. Filter to First NFL Season

**Critical temporal boundary**: We only use outcomes from the player's FIRST NFL season.
This ensures a clean prospective scouting design.

In [ ]:
# Try to determine season column
season_col = None
for col in ['season', 'year', 'gameYear']:
    if col in player_play.columns:
        season_col = col
        break

# If no season column, try extracting from game_id
if season_col is None and 'game_id' in player_play.columns:
    print('Extracting season from game_id...')
    sample_game_id = player_play['game_id'].dropna().iloc[0] if len(player_play) > 0 else ''
    print(f'Sample game_id: {sample_game_id}')
    # Common format: "2023_01_..." or "2023010100"
    player_play_copy = player_play.copy()
    player_play_copy['season'] = player_play_copy['game_id'].astype(str).str[:4]
    # Validate it looks like a year
    year_vals = player_play_copy['season'].unique()
    print(f'Extracted year values: {year_vals[:10]}')
    if all(str(y).isdigit() and 2000 <= int(y) <= 2030 for y in year_vals if str(y) != 'nan'):
        player_play = player_play_copy
        season_col = 'season'
        print(f'Season column created: {player_play["season"].value_counts().to_dict()}')
    else:
        print('Could not reliably extract season from game_id')

print(f'Season column used: {season_col}')

In [ ]:
# Filter to first season
if season_col is not None:
    player_play_first = filter_to_first_season(player_play, season_col=season_col)
    print(f'First-season plays: {len(player_play_first):,} rows | {player_play_first["nfl_id"].nunique()} players')
else:
    print('WARNING: Could not identify season column — using ALL plays (not first season only)')
    print('This may include multiple seasons. Interpret results with caution.')
    player_play_first = player_play

# Add position to player_play
player_play_first = player_play_first.merge(pos_map, on='nfl_id', how='left')

print(f'\nPosition distribution in first-season plays:')
if 'position' in player_play_first.columns:
    display(player_play_first['position'].value_counts().to_frame())

## 4. Aggregate NFL Metrics to Player Level

In [ ]:
nfl_metrics = aggregate_nfl_metrics(
    player_play=player_play_first,
    players=players,
    season_col=season_col,
    first_season_only=False,  # Already filtered above
)
print(f'Player-level NFL metrics: {nfl_metrics.shape}')
display(nfl_metrics.head())

# Snap counts (important confounder)
snap_counts = compute_snap_counts(player_play_first, season_col=season_col)
nfl_metrics = nfl_metrics.merge(snap_counts.rename(columns={'n_snaps': 'first_season_snaps'}), on='nfl_id', how='left')
print(f'\nSnap count distribution:')
display(nfl_metrics['first_season_snaps'].describe())

# Save
nfl_metrics.to_parquet(DATA_PROCESSED / 'nfl_metrics.parquet', index=False)
print('Saved: nfl_metrics.parquet')

## 5. Build Analysis Dataset

Join combine features + NFL metrics at the player level.
This is the core dataset for hypothesis discovery.

In [ ]:
if not player_features.empty:
    # Pivot combine features wide (one row per player, columns per drill)
    feat_cols = [c for c in player_features.columns if c not in ['nfl_id', 'drill_type', 'attempt', 'position', 'position_group']]
    if 'drill_type' in player_features.columns:
        feat_wide = player_features.pivot_table(
            index='nfl_id', columns='drill_type', values=feat_cols, aggfunc='first'
        )
        feat_wide.columns = ['_'.join(map(str, c)).strip() for c in feat_wide.columns]
        feat_wide = feat_wide.reset_index()
    else:
        feat_wide = player_features.drop(columns=['attempt'], errors='ignore')

    # Join
    analysis_df = nfl_metrics.merge(feat_wide, on='nfl_id', how='inner')
    analysis_df = analysis_df.merge(pos_map, on='nfl_id', how='left', suffixes=('', '_pos'))

    print(f'Analysis dataset: {analysis_df.shape}')
    print(f'Unique players: {analysis_df["nfl_id"].nunique()}')
    print(f'Position groups present: {analysis_df["position_group"].value_counts().to_dict() if "position_group" in analysis_df.columns else "N/A"}')

    analysis_df.to_parquet(DATA_PROCESSED / 'analysis_dataset.parquet', index=False)
    print('Saved: analysis_dataset.parquet')
else:
    print('Combine features not available — run notebook 02 first.')
    analysis_df = pd.DataFrame()

## 6. Correlation Discovery: Combine → NFL Outcomes

For each position group, compute Spearman correlations between
combine movement features and NFL outcome metrics.

In [ ]:
# Define which outcome metrics are relevant per position group
POSITION_OUTCOME_MAP = {
    'WR':    ['mean_separation', 'median_separation', 'mean_yac', 'yac_over_expected', 'target_share'],
    'TE':    ['mean_separation', 'mean_yac', 'yards_per_reception', 'target_share'],
    'Edge':  ['mean_get_off', 'quick_pressure_rate', 'sack_rate', 'mean_time_to_pressure'],
    'DT':    ['quick_pressure_rate', 'sack_rate', 'n_tfl'],
    'OL':    ['pressure_allowed_rate', 'sack_allowed_rate'],
    'CB':    ['n_tackles'],
    'S':     ['n_tackles'],
    'RB':    ['mean_yac', 'yards_per_reception'],
}

all_correlations = []

if not analysis_df.empty and 'position_group' in analysis_df.columns:
    for pos_group, outcomes in POSITION_OUTCOME_MAP.items():
        group_data = analysis_df[analysis_df['position_group'] == pos_group].copy()
        n = group_data['nfl_id'].nunique()

        if n < MIN_SAMPLE_SIZE:
            print(f'{pos_group}: SKIP (n={n} < {MIN_SAMPLE_SIZE})')
            continue

        print(f'\n=== {pos_group} (n={n}) ===')

        # Available combine features for this group
        combine_feats = [c for c in group_data.columns
                         if any(c.startswith(f) for f in [
                             'max_speed', 'first_step', 'total_direction', 'movement_efficiency',
                             'best_peak_reaccel', 'time_to_75pct', 'direction_change_rate',
                             'n_cut_sequences', 'mean_deceleration', 'burst_impulse',
                             'mean_speed',
                         ])]

        avail_outcomes = [c for c in outcomes if c in group_data.columns]

        if not combine_feats or not avail_outcomes:
            print(f'  No valid features/outcomes available')
            continue

        corr_summary = build_correlation_summary(
            group_data, combine_feats, avail_outcomes, method='spearman', min_n=MIN_SAMPLE_SIZE
        )
        corr_summary['position_group'] = pos_group
        all_correlations.append(corr_summary)

        if not corr_summary.empty:
            print(f'Top correlations (n={n}):')
            display(corr_summary.head(10))
else:
    print('Analysis dataset not available or no position_group column.')

In [ ]:
# Combine all correlations into one ranked table
if all_correlations:
    master_corr = pd.concat(all_correlations, ignore_index=True)
    master_corr = master_corr.sort_values('abs_rho', ascending=False)
    
    print('\n=== TOP 20 COMBINE → NFL CORRELATIONS (across all positions) ===')
    display(master_corr.head(20))

    # Save
    master_corr.to_parquet(DATA_PROCESSED / 'correlation_summary.parquet', index=False)
    master_corr.to_csv(TABLES_DIR / 'correlation_summary.csv', index=False)
    print('Saved correlation summary.')

    # Visualize top correlations
    top_pos_groups = master_corr['position_group'].unique()[:3]
    for pos in top_pos_groups:
        pos_corr = master_corr[master_corr['position_group'] == pos].head(5)
        if len(pos_corr) == 0:
            continue
        for _, row in pos_corr.iterrows():
            plot_combine_vs_nfl(
                analysis_df[analysis_df['position_group'] == pos],
                x_col=row['combine_feature'],
                y_col=row['nfl_outcome'],
                title=f'{pos}: {row["combine_feature"]} vs {row["nfl_outcome"]}\nρ={row["rho"]:.3f} (n={row["n"]})',
                filename=f'scatter_{pos}_{row["combine_feature"]}_vs_{row["nfl_outcome"]}.png'.replace('/', '_'),
            )
            plt.show()
else:
    print('NOT YET COMPUTED — no correlations available.')

## 7. Incremental Value: Tracking vs Traditional

Does tracking data add value beyond traditional combine metrics?

In [ ]:
if not analysis_df.empty:
    traditional_cols = [c for c in ['ten_yd_split', 'forty', 'vertical', 'broad_jump',
                                     'three_cone', 'short_shuttle', 'ngs_athleticism_score']
                        if c in analysis_df.columns]

    tracking_cols = [c for c in analysis_df.columns
                     if any(c.startswith(f) for f in [
                         'max_speed', 'first_step', 'total_direction', 'best_peak_reaccel',
                         'movement_efficiency', 'direction_change_rate',
                     ])]

    print(f'Traditional features: {traditional_cols}')
    print(f'Tracking features: {len(tracking_cols)}')

    # Test on WR separation as primary target
    if 'mean_separation' in analysis_df.columns and traditional_cols and tracking_cols:
        wr_data = analysis_df[analysis_df['position_group'] == 'WR'].copy()

        if len(wr_data) >= MIN_SAMPLE_SIZE:
            y = wr_data['mean_separation'].values
            xb = wr_data[traditional_cols]
            xf = wr_data[traditional_cols + tracking_cols[:10]]  # limit features

            result = incremental_r_squared(y, xb, xf)
            print(f'\nWR separation — incremental R² analysis:')
            for k, v in result.items():
                print(f'  {k}: {v}')

            plot_model_comparison(
                {
                    'Traditional Only': result.get('r2_baseline', 0),
                    'Traditional + Tracking': result.get('r2_full', 0),
                },
                title='Predicting WR Separation: Traditional vs Tracking-Enhanced',
                filename='model_comparison_wr_separation.png',
            )
            plt.show()
        else:
            print(f'Insufficient WR data for incremental R² analysis (n={len(wr_data)})')
    else:
        print('NOT YET COMPUTED — required columns not available')
else:
    print('Analysis dataset not available.')

## 8. Rank Hypothesis Candidates

Synthesize all findings into a ranked list of research directions.

In [ ]:
# This cell summarizes hypothesis candidates based on actual data above.
# The template below will be auto-populated where data is available,
# and flagged as NOT YET COMPUTED where it is not.

def format_hypothesis_report(all_correlations_df: pd.DataFrame) -> str:
    """Generate hypothesis candidates markdown from correlation summary."""
    lines = []
    lines.append('# NFL Big Data Bowl 2027 — Hypothesis Candidates')
    lines.append(f'\n*Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}*')
    lines.append('\n> All numbers below come from actual data. Where data was unavailable, "NOT YET COMPUTED" is stated explicitly.\n')

    if all_correlations_df is not None and not all_correlations_df.empty:
        # Get top unique candidates
        top = all_correlations_df.head(10)

        lines.append('## Ranked Correlation Findings')
        lines.append('Ranked by |ρ| (Spearman). Minimum n = 10 players.\n')
        lines.append('| Rank | Position | Combine Feature | NFL Outcome | ρ | n | p-value |')
        lines.append('|------|----------|-----------------|-------------|---|---|---------|')

        for i, row in top.iterrows():
            lines.append(
                f'| {i+1} | {row["position_group"]} | {row["combine_feature"]} | '
                f'{row["nfl_outcome"]} | {row["rho"]:.3f} | {row["n"]} | {row["p_value"]:.4f} |'
            )
    else:
        lines.append('## Ranked Correlation Findings')
        lines.append('\n**NOT YET COMPUTED** — Run all cells above with actual data.')

    lines.append('\n\n## Candidate Hypotheses (Template)')
    lines.append('\nFill in based on actual data findings above.\n')

    hypotheses_template = [
        {
            'rank': 1,
            'hypothesis': 'NOT YET DETERMINED — pending data analysis',
            'position': 'TBD',
            'combine_drill': 'TBD',
            'tracking_metric': 'TBD',
            'nfl_outcome': 'TBD',
            'sample_size': 'TBD',
            'evidence': 'NOT YET COMPUTED',
            'novelty': 'TBD',
            'football_interpretation': 'TBD',
            'potential_confounders': 'draft position, team scheme, snap count, route type',
            'difficulty': 'TBD',
            'competition_potential': 'TBD',
        },
    ]

    # If we have real correlations, populate top candidates
    if all_correlations_df is not None and not all_correlations_df.empty:
        hypotheses_template = []
        top5 = all_correlations_df.head(5)
        for rank, (_, row) in enumerate(top5.iterrows(), 1):
            sig_note = '(statistically significant)' if row['p_value'] < 0.05 else '(not yet significant — needs more data)'
            hypotheses_template.append({
                'rank': rank,
                'hypothesis': (
                    f'{row["combine_feature"]} during Combine drills is associated with '
                    f'{row["nfl_outcome"]} in {row["position_group"]} players'
                ),
                'position': row['position_group'],
                'tracking_metric': row['combine_feature'],
                'nfl_outcome': row['nfl_outcome'],
                'sample_size': row['n'],
                'evidence': f'ρ = {row["rho"]:.3f}, p = {row["p_value"]:.4f} {sig_note}',
                'potential_confounders': 'draft position, team scheme, snap count, opponent quality, position within scheme',
            })

    for hyp in hypotheses_template:
        lines.append(f'\n### Hypothesis #{hyp["rank"]}')
        for key, val in hyp.items():
            if key != 'rank':
                lines.append(f'- **{key.replace("_", " ").title()}**: {val}')

    lines.append('\n\n## Important Caveats')
    lines.append('\n- Correlation ≠ causation. All findings are associations.')
    lines.append('- Sample sizes are small (~510 players). Effect sizes may be unstable.')
    lines.append('- Tracking frames are NOT independent observations. The player is the unit of analysis.')
    lines.append('- Confounders (team, scheme, snap count, opponent) are not yet controlled.')
    lines.append('- First-season performance may not reflect long-term potential.')

    return '\n'.join(lines)


all_corr_df = None
if all_correlations:
    all_corr_df = pd.concat(all_correlations, ignore_index=True).sort_values('abs_rho', ascending=False)

report_text = format_hypothesis_report(all_corr_df)
report_path = REPORTS_DIR / 'hypothesis_candidates.md'
report_path.write_text(report_text, encoding='utf-8')
print(f'Saved: {report_path}')
print()
print(report_text)

## 9. Phase 1 Summary

Fill in after running all cells. Do NOT fabricate values.

In [ ]:
print('=== PHASE 1 SUMMARY ===')
print()

# Dataset statistics
print('--- Dataset ---')
if 'players' in dir() and players is not None:
    print(f'Players: {len(players):,}')
if 'player_features' in dir() and not player_features.empty:
    print(f'Combine attempts: {len(player_features):,}')
    if 'drill_type' in player_features.columns:
        print(f'Drills: {player_features["drill_type"].nunique()}')
if 'player_play_first' in dir() and player_play_first is not None and season_col is not None:
    print(f'NFL seasons: {player_play_first[season_col].nunique() if season_col in player_play_first.columns else "N/A"}')
print()

# Most promising
print('--- Most Promising (based on data above) ---')
if all_corr_df is not None and not all_corr_df.empty:
    top_positions = all_corr_df.groupby('position_group')['abs_rho'].max().sort_values(ascending=False).head(3)
    print('\nTop position groups:')
    for i, (pos, rho) in enumerate(top_positions.items(), 1):
        print(f'  {i}. {pos} (max |ρ| = {rho:.3f})')

    top_feats = all_corr_df.groupby('combine_feature')['abs_rho'].max().sort_values(ascending=False).head(5)
    print('\nTop tracking features:')
    for i, (feat, rho) in enumerate(top_feats.items(), 1):
        print(f'  {i}. {feat} (max |ρ| = {rho:.3f})')

    top_outcomes = all_corr_df.groupby('nfl_outcome')['abs_rho'].max().sort_values(ascending=False).head(3)
    print('\nTop NFL outcomes:')
    for i, (out, rho) in enumerate(top_outcomes.items(), 1):
        print(f'  {i}. {out} (max |ρ| = {rho:.3f})')
else:
    print('NOT YET COMPUTED — run all cells above with actual data.')

print('\n--- Recommended Primary Research Direction ---')
print('NOT YET DETERMINED — review hypothesis_candidates.md after running with data.')